# Notebook 03 — Train / Validation / Test Split

## Objective

The goal of this notebook is to split the labeled order-level dataset into training, validation, and test sets.


A chronological split is used so the model is trained on older orders and evaluated on newer orders, which better represents the real prediction scenario.

### Input Artifact

`artifacts/02_labeled/labeled_orders.parquet`

### Output Artifacts

- `artifacts/03_splits/train.parquet`
- `artifacts/03_splits/validation.parquet`
- `artifacts/03_splits/test.parquet`

## Imports & Paths

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = PROJECT_ROOT / "artifacts" / "02_labeled" / "labeled_orders.parquet"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "03_splits"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_parquet(INPUT_PATH)

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

print("Shape:", df.shape)
print("Date range:")
print(df["order_purchase_timestamp"].min())
print(df["order_purchase_timestamp"].max())

Shape: (96470, 39)
Date range:
2016-09-15 12:16:38
2018-08-29 15:00:37


## Splitting using 70% / 15% / 15% 

In [5]:
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (67529, 39)
Validation: (14470, 39)
Test: (14471, 39)


In [6]:
for name, split in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(
        "Dates:",
        split["order_purchase_timestamp"].min(),
        "to",
        split["order_purchase_timestamp"].max()
    )
    
    print(
        split["is_late"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )


Train
Dates: 2016-09-15 12:16:38 to 2018-04-15 20:12:35
is_late
0    90.97
1     9.03
Name: proportion, dtype: float64

Validation
Dates: 2018-04-15 20:17:11 to 2018-06-21 08:29:29
is_late
0    94.66
1     5.34
Name: proportion, dtype: float64

Test
Dates: 2018-06-21 08:41:07 to 2018-08-29 15:00:37
is_late
0    93.39
1     6.61
Name: proportion, dtype: float64


In [7]:
assert len(train_df) + len(val_df) + len(test_df) == len(df)

assert set(train_df["order_id"]).isdisjoint(val_df["order_id"])
assert set(train_df["order_id"]).isdisjoint(test_df["order_id"])
assert set(val_df["order_id"]).isdisjoint(test_df["order_id"])

assert train_df["order_purchase_timestamp"].max() <= val_df["order_purchase_timestamp"].min()
assert val_df["order_purchase_timestamp"].max() <= test_df["order_purchase_timestamp"].min()

print("Split validation passed.")

Split validation passed.



### Split Check

The chronological split preserves the temporal order of the data.

The late-delivery rate varies across the splits:
- Train: 9.03%
- Validation: 5.34%
- Test: 6.61%

This difference is expected with a time-based split and may reflect changes in delivery performance over time. The class imbalance will be handled during modeling using the training data only.

In [9]:
assert len(train_df) + len(val_df) + len(test_df) == len(df)

assert set(train_df["order_id"]).isdisjoint(val_df["order_id"])
assert set(train_df["order_id"]).isdisjoint(test_df["order_id"])
assert set(val_df["order_id"]).isdisjoint(test_df["order_id"])

print("Split validation passed.")

Split validation passed.


In [10]:
train_df.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
val_df.to_parquet(OUTPUT_DIR / "validation.parquet", index=False)
test_df.to_parquet(OUTPUT_DIR / "test.parquet", index=False)

print("Artifacts saved successfully.")

Artifacts saved successfully.
